# 540 - Token Introduction

A token is a piece of text: a word, part of a word, a space, or punctuation. A token ID is the integer assigned to that piece by a particular tokenizer.

Run the cells in order. Compare words and tokens, then recover the text from the tokens.

Use the existing notebook environment. The next cell installs only the libraries required for this exercise.

In [ ]:
%pip install -q "google-genai==2.25.0" "sentencepiece==0.2.1" "protobuf>=5,<8"

## 1. Split a sentence into words

`split()` separates text at whitespace. It does not perform model tokenization; punctuation stays attached to words.

In [ ]:
text = "Welcome to Python and AI with simple code, clear ideas, and practical projects for real work."
words = text.split()
print("Text:", text)
print("Words:", words)
print("Word count:", len(words))

## 2. Load a Gemini tokenizer

Use the tokenizer for `gemini-2.5-flash`, matching the model in the other notebooks. Only tokenizer data is downloaded on the first run; the generation model is not downloaded. No Google Cloud login or API key is needed here.

Google's local tokenizer is experimental and supports text only. Internet access is needed for the first download. Restart the kernel after installation if prompted.

For the lighter Flash-Lite model, change `TOKENIZER_MODEL` to `gemini-2.5-flash-lite`. These two models use the same local tokenizer in this SDK, so their local token IDs match.

In [ ]:
from google.genai import local_tokenizer

TOKENIZER_MODEL = "gemini-2.5-flash"
tokenizer = local_tokenizer.LocalTokenizer(model_name=TOKENIZER_MODEL)
print("Tokenizer model:", TOKENIZER_MODEL)

## 3. Get and print tokens

`compute_tokens()` returns token IDs and their corresponding byte pieces. `repr()` makes spaces visible. One word can span several tokens, and punctuation can be a separate token.

In [ ]:
result = tokenizer.compute_tokens(text)
info = result.tokens_info[0]
token_ids = info.token_ids
token_bytes = info.tokens

print("Token IDs:", token_ids)
print("Word count:", len(words))
print("Token count:", len(token_ids))
print("\nToken ID -> text piece")
for token_id, piece in zip(token_ids, token_bytes):
    print(token_id, "->", repr(piece.decode("utf-8", errors="replace")))

## 4. Recover text and words from the tokens

The SDK returns IDs and pieces together; it has no public `decode(token_ids)` method. Build a small lookup from the returned pairs and turn these IDs back into bytes. This lookup covers this sentence, not the entire vocabulary.

Join bytes before decoding UTF-8 because a character can span multiple tokens. This reconstructs the input; it does not ask Gemini to generate an answer.

In [ ]:
id_to_piece = dict(zip(token_ids, token_bytes))
recovered_bytes = b"".join(id_to_piece[token_id] for token_id in token_ids)
recovered_text = recovered_bytes.decode("utf-8")

print("Recovered text:", repr(recovered_text))
print("Recovered words:", recovered_text.split())
print("Same text:", recovered_text == text)
assert recovered_text == text

## 5. Compare a few strings

There is no fixed number of tokens per word. Token IDs belong to a particular tokenizer, so do not reuse them with an unrelated model.

In [ ]:
examples = [
    "Welcome to Python",
    "Welcome to Python!",
    "Tokenization turns text into tokens.",
]
for sample in examples:
    sample_info = tokenizer.compute_tokens(sample).tokens_info[0]
    print("\nText:", sample)
    print("Words:", len(sample.split()), "Tokens:", len(sample_info.token_ids))
    print("Token IDs:", sample_info.token_ids)
    print("Pieces:", sample_info.tokens)

## 6. Optional: count tokens through your Google Cloud project

Use the existing Google Cloud configuration and Application Default Credentials (ADC). This cell makes one Vertex AI token-count request.

`count_tokens()` returns a count, not token pieces. The cloud count can differ from the local text-piece count. An embedding model is not needed here: embeddings represent meaning as vectors and are not a reversible list of text tokens.

In [ ]:
RUN_CLOUD_COUNT = False  # Set True to run the optional cloud count.

if RUN_CLOUD_COUNT:
    from course_config import PROJECT_ID, LOCATION, MODEL_ID, make_client

    print("Project:", PROJECT_ID, "| Location:", LOCATION, "| Model:", MODEL_ID)
    with make_client() as client:
        cloud_count = client.models.count_tokens(model=MODEL_ID, contents=text)
    print("Cloud token count:", cloud_count.total_tokens)
    print("Local token count:", len(token_ids))
else:
    print("Optional cloud count is off. The local exercise is complete.")

## Try it

Change the main sentence to another 10-20 word sentence and rerun sections 1-5. Add punctuation or change a word to see how the pieces change.

References: [Google Gen AI SDK](https://googleapis.github.io/python-genai/) and [List and count tokens](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/capabilities/list-token).